# 08 - Bureau Master Data Creation

**Objective:** Create a unified bureau master file from 5 CRIF bureau data files (Summary, Account, IOI, Inquiry, Demog) across 2 sets.

**Grain:** `LOS-APP-ID × Bureau Account (trade line)` — one row per applicant per bureau trade line.

- For each **loan** there are multiple **applicants** (O01=Primary, O02=Co-applicant, O03=Guarantor)
- For each **applicant** there are multiple **bureau trade lines** (loans from other lenders)
- Summary/Inquiry/IOI/Demog features are joined at the LOS-APP-ID level

**Output:** `bureau_data/bureau_master_account_level.parquet` + `.csv` + report

## Section 0: Setup

In [3]:
import pandas as pd
import numpy as np
import os
import re
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 80)

# Paths
BASE_DIR = '/Users/shiva/Downloads/Five_star_customer_data'
BUREAU_DIR = os.path.join(BASE_DIR, 'bureau_data', 'bureau_data1')
SET1_DIR = os.path.join(BUREAU_DIR, 'FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set1_output')
SET2_DIR = os.path.join(BUREAU_DIR, 'FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set2_output')
OUTPUT_DIR = os.path.join(BASE_DIR, 'bureau_data')
REPORT_DIR = os.path.join(BASE_DIR, 'eda_reports')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print('Set1 dir exists:', os.path.exists(SET1_DIR))
print('Set2 dir exists:', os.path.exists(SET2_DIR))

Set1 dir exists: True
Set2 dir exists: True


In [12]:
# ============================================================
# Helper Functions
# ============================================================

def load_bureau_csv(filepath, usecols=None, nrows=None, dtype=None):
    """Load pipe-delimited bureau CSV with BOM encoding."""
    return pd.read_csv(
        filepath, sep='|', encoding='utf-8-sig', quotechar='"',
        usecols=usecols, nrows=nrows, dtype=dtype,
        on_bad_lines='skip'
    )


def indian_comma_to_numeric(series):
    """Convert Indian comma-formatted strings to float. e.g. '3,00,000' -> 300000.0"""
    if hasattr(series, 'str'):
        return pd.to_numeric(series.str.replace(',', '', regex=False), errors='coerce')
    return pd.to_numeric(series, errors='coerce')


def parse_age_string(series):
    """Convert 'Xyrs Ymon' to total months. e.g. '1yrs 7mon' -> 19. None -> NaN."""
    original_null = series.isna()
    years = series.str.extract(r'(\d+)yrs', expand=False).astype(float)
    months = series.str.extract(r'(\d+)mon', expand=False).astype(float)
    total = (years.fillna(0) * 12 + months.fillna(0))
    total[original_null] = pd.NA
    return total.astype('Int64')


def parse_dpd_history(dpd_string):
    """Parse DPD history string into list of ints. 3-char groups, XXX=NaN.
    First group = most recent month.
    e.g. '000090XXX' -> [0, 0, 90, NaN]
    """
    if pd.isna(dpd_string) or not isinstance(dpd_string, str) or len(dpd_string.strip()) == 0:
        return []
    s = dpd_string.strip()
    groups = [s[i:i+3] for i in range(0, len(s), 3)]
    result = []
    for g in groups:
        if len(g) < 3:
            continue
        if g == 'XXX':
            result.append(np.nan)
        else:
            try:
                result.append(int(g))
            except ValueError:
                result.append(np.nan)
    return result

import pandas as pd

def calculate_fill_rates(df):
    total_rows = len(df)
    
    fill_rate = pd.DataFrame({
        'Column': df.columns,
        'Total Rows': total_rows,
        'Non-Null Count': df.count().values,
        'Null Count': df.isnull().sum().values,
        'Fill Rate (%)': (df.count().values / total_rows * 100).round(2)
    })
    
    fill_rate = fill_rate.sort_values('Fill Rate (%)', ascending=False).reset_index(drop=True)
    
    return fill_rate


def parse_installment_amt(series):
    """Parse 'amount/frequency' format. e.g. '7,356/Monthly' -> 7356.0"""
    if not hasattr(series, 'str'):
        return pd.to_numeric(series, errors='coerce')
    amt_part = series.str.split('/', n=1).str[0]
    return indian_comma_to_numeric(amt_part)


def extract_los_parts(df):
    """Extract loan_number, applicant_type, person_id from LOS-APP-ID."""
    df['loan_number'] = df['LOS-APP-ID'].str.extract(r'^(.+?)_O0[123]_')
    df['applicant_type'] = df['LOS-APP-ID'].str.extract(r'_(O0[123])_')
    df['person_id'] = df['LOS-APP-ID'].str.extract(r'_O0[123]_(\d+)$')
    return df


print('Helper functions defined.')

Helper functions defined.


In [7]:
# ============================================================
# QC Helper Functions
# ============================================================

# indian_comma_to_numeric
test_s = pd.Series(['3,00,000', '49,007', '0', '', None, '1,23,45,678'])
result = indian_comma_to_numeric(test_s)
assert result[0] == 300000.0, f'Expected 300000, got {result[0]}'
assert result[1] == 49007.0, f'Expected 49007, got {result[1]}'
assert result[2] == 0.0
assert pd.isna(result[4])
print('indian_comma_to_numeric: PASSED')

# parse_age_string
test_s2 = pd.Series(['1yrs 7mon', '4yrs 0mon', '0yrs 6mon', None])
result2 = parse_age_string(test_s2)
assert result2[0] == 19, f'Expected 19, got {result2[0]}'
assert result2[1] == 48, f'Expected 48, got {result2[1]}'
assert result2[2] == 6
assert pd.isna(result2[3])
print('parse_age_string: PASSED')

# parse_dpd_history
# '000090XXX' = 3 groups: 000, 090, XXX -> [0, 90, NaN]
dpd_result = parse_dpd_history('000090XXX')
assert len(dpd_result) == 3, f'Expected 3 groups, got {len(dpd_result)}'
assert dpd_result[0] == 0 and dpd_result[1] == 90 and np.isnan(dpd_result[2])
# Longer test: '000000000000090900XXX' = 7 groups
dpd_result2 = parse_dpd_history('000000000000090900XXX')
assert len(dpd_result2) == 7
assert dpd_result2[0] == 0 and dpd_result2[4] == 90 and dpd_result2[5] == 900
assert parse_dpd_history('') == []
assert parse_dpd_history(None) == []
assert parse_dpd_history('000000000') == [0, 0, 0]
print('parse_dpd_history: PASSED')

# parse_installment_amt
test_s3 = pd.Series(['7,356/Monthly', '10,120', None, '0'])
result3 = parse_installment_amt(test_s3)
assert result3[0] == 7356.0, f'Expected 7356, got {result3[0]}'
assert result3[1] == 10120.0
assert result3[3] == 0.0
print('parse_installment_amt: PASSED')

# extract_los_parts
test_df = pd.DataFrame({'LOS-APP-ID': ['FSAPLALONS000005569308_O01_5050883660', 'FSPYLALONS000005273350_O02_5960284496', 'FSAPLALONS000005492830_O03_5960742851']})
test_df = extract_los_parts(test_df)
assert test_df['loan_number'].iloc[0] == 'FSAPLALONS000005569308'
assert test_df['applicant_type'].iloc[1] == 'O02'
assert test_df['person_id'].iloc[2] == '5960742851'
print('extract_los_parts: PASSED')

print('\nAll helper QC tests PASSED.')

indian_comma_to_numeric: PASSED
parse_age_string: PASSED
parse_dpd_history: PASSED
parse_installment_amt: PASSED
extract_los_parts: PASSED

All helper QC tests PASSED.


## Section 1: Summary — Applicant-Level Credit Scores & Portfolio Summary

In [8]:
# Load and concat Summary set1 + set2
summary1 = load_bureau_csv(os.path.join(SET1_DIR, 'Summary.csv'))
summary2 = load_bureau_csv(os.path.join(SET2_DIR, 'Summary.csv'))

print(f'Summary set1: {summary1.shape}')
print(f'Summary set2: {summary2.shape}')

summary = pd.concat([summary1, summary2], ignore_index=True)
del summary1, summary2

# Drop trailing empty column from pipe delimiter
summary = summary.loc[:, ~(summary.columns.str.strip() == '')]

print(f'Summary combined: {summary.shape}')
print(f'\nSTATUS distribution:\n{summary["STATUS"].value_counts()}')
print(f'\nDuplicates on LOS-APP-ID: {summary["LOS-APP-ID"].duplicated().sum()}')

Summary set1: (999998, 29)
Summary set2: (550373, 29)
Summary combined: (1550371, 29)

STATUS distribution:
STATUS
SUCCESS    1550368
ERROR            3
Name: count, dtype: int64

Duplicates on LOS-APP-ID: 0


In [9]:
# Filter to SUCCESS, clean up
summary = summary[summary['STATUS'] == 'SUCCESS'].copy()
print(f'After STATUS=SUCCESS filter: {summary.shape}')

# Drop non-useful columns
drop_cols = ['CUSTOMER ID/MBR ID', 'CREDT-RPT-ID', 'STATUS', 'ERROR']
drop_cols = [c for c in drop_cols if c in summary.columns]
summary.drop(columns=drop_cols, inplace=True)

# Convert numeric columns (Indian comma format)
numeric_cols_summary = [
    'PERFORM_CNS SCORE', 'PRI-NO-OF-ACCTS', 'PRI-ACTIVE-ACCTS', 'PRI-OVERDUE-ACCTS',
    'PRI-CURRENT-BALANCE', 'PRI-SANCTIONED-AMOUNT', 'PRI-DISBURSED-AMOUNT',
    'SEC-NO-OF-ACCTS', 'SEC-ACTIVE-ACCTS', 'SEC-OVERDUE-ACCTS',
    'SEC-CURRENT-BALANCE', 'SEC-SANCTIONED-AMOUNT', 'SEC-DISBURSED-AMOUNT',
    'PRIMARY-INSTAL-AMT', 'SEC-INSTAL-AMT',
    'NEW-ACCTS-IN-LAST-SIX-MONTHS', 'DELINQUENT-ACCTS-IN-LAST-SIX-MONTHS',
    'NO-OF_INQUIRIES'
]
for col in numeric_cols_summary:
    if col in summary.columns:
        summary[col] = indian_comma_to_numeric(summary[col])

# Parse age strings to months
if 'AVERAGE-ACCT-AGE' in summary.columns:
    summary['avg_acct_age_months'] = parse_age_string(summary['AVERAGE-ACCT-AGE'])
    summary.drop(columns=['AVERAGE-ACCT-AGE'], inplace=True)

if 'CREDIT-HISTORY-LENGTH' in summary.columns:
    summary['credit_history_length_months'] = parse_age_string(summary['CREDIT-HISTORY-LENGTH'])
    summary.drop(columns=['CREDIT-HISTORY-LENGTH'], inplace=True)

# Rename for clarity, add prefix
rename_map = {col: f'summary_{col.lower().replace(" ", "_").replace("-", "_")}' 
              for col in summary.columns if col != 'LOS-APP-ID'}
# Keep avg_acct_age_months and credit_history_length_months as-is (already clean)
rename_map = {k: v for k, v in rename_map.items() 
              if k not in ['avg_acct_age_months', 'credit_history_length_months']}
summary.rename(columns=rename_map, inplace=True)

print(f'Summary cleaned: {summary.shape}')
print(f'Columns: {list(summary.columns)}')
summary.head(2)

After STATUS=SUCCESS filter: (1550368, 29)
Summary cleaned: (1550368, 25)
Columns: ['LOS-APP-ID', 'summary_perform_cns_score', 'summary_perform_cns_score_description', 'summary_income_band', 'summary_income_band_description', 'summary_pri_no_of_accts', 'summary_pri_active_accts', 'summary_pri_overdue_accts', 'summary_pri_current_balance', 'summary_pri_sanctioned_amount', 'summary_pri_disbursed_amount', 'summary_sec_no_of_accts', 'summary_sec_active_accts', 'summary_sec_overdue_accts', 'summary_sec_current_balance', 'summary_sec_sanctioned_amount', 'summary_sec_disbursed_amount', 'summary_primary_instal_amt', 'summary_sec_instal_amt', 'summary_new_accts_in_last_six_months', 'summary_delinquent_accts_in_last_six_months', 'summary_no_of_inquiries', 'summary_unnamed:_28', 'avg_acct_age_months', 'credit_history_length_months']


,LOS-APP-ID,summary_perform_cns_score,summary_perform_cns_score_description,summary_income_band,summary_income_band_description,summary_pri_no_of_accts,summary_pri_active_accts,summary_pri_overdue_accts,summary_pri_current_balance,summary_pri_sanctioned_amount,summary_pri_disbursed_amount,summary_sec_no_of_accts,summary_sec_active_accts,summary_sec_overdue_accts,summary_sec_current_balance,summary_sec_sanctioned_amount,summary_sec_disbursed_amount,summary_primary_instal_amt,summary_sec_instal_amt,summary_new_accts_in_last_six_months,summary_delinquent_accts_in_last_six_months,summary_no_of_inquiries,summary_unnamed:_28,avg_acct_age_months,credit_history_length_months
0,FSAPLALONS000005569308_O01_5050883660,628,NaN,C,"3,00,001 - 4,00,000",1.0,1.0,1.0,286029.0,350000.0,350000.0,0.0,0.0,0.0,0.0,0.0,0.0,10120.0,0.0,0.0,1.0,0.0,NaN,19,19
1,FSAPLALONS000005492830_O03_5960742851,695,NaN,D,"4,00,001 - 6,00,000",38.0,3.0,1.0,714311.0,850046.0,850046.0,0.0,0.0,0.0,0.0,0.0,0.0,5783.0,0.0,15.0,0.0,0.0,NaN,6,48


## Section 2: Account Data — Bureau Trade Lines (Granular, Chunked)

In [10]:
# Define Account columns to keep and their processing
ACCOUNT_COLS_KEEP = [
    'LOS-APP-ID', 'CANDIDATE - ID', 'SELF-INDICATOR', 'MATCH-TYPE',
    'ACC-NUM', 'CREDIT-GRANTOR', 'ACCT-TYPE', 'CONTRIBUTOR-TYPE',
    'DATE-REPORTED', 'OWNERSHIP-IND', 'ACCOUNT-STATUS',
    'DISBURSED-DT', 'CLOSE-DT', 'LAST-PAYMENT-DATE',
    'CREDIT-LIMIT/SANC AMT', ' DISBURSED-AMT/HIGH CREDIT', ' INSTALLMENT-AMT',
    ' CURRENT-BAL', 'INSTALLMENT-FREQUENCY', ' OVERDUE-AMT', ' WRITE-OFF-AMT',
    'ASSET_CLASS', ' ACCOUNT-REMARKS',
    'DPD - HIST', 'REPORTED DATE - HIST',
    'WRITE-OFF-DATE', 'INCOME ', ' INCOME INDICATOR ', 'TENURE ', ' OCCUPATION'
]

# Numeric columns in Account (Indian comma format)
ACCOUNT_NUMERIC_COLS = [
    'CREDIT-LIMIT/SANC AMT', ' DISBURSED-AMT/HIGH CREDIT',
    ' CURRENT-BAL', ' OVERDUE-AMT', ' WRITE-OFF-AMT', 'INCOME '
]

# Date columns
ACCOUNT_DATE_COLS = ['DISBURSED-DT', 'CLOSE-DT', 'LAST-PAYMENT-DATE', 'DATE-REPORTED', 'WRITE-OFF-DATE']

print(f'Will keep {len(ACCOUNT_COLS_KEEP)} columns from Account data')

Will keep 30 columns from Account data


In [11]:
def process_account_chunk(chunk):
    """Process a chunk of Account data: clean numerics, parse dates, extract DPD features."""
    # Drop trailing empty column
    chunk = chunk.loc[:, ~(chunk.columns.str.strip() == '')]
    
    # Keep only desired columns (handle missing cols gracefully)
    cols_available = [c for c in ACCOUNT_COLS_KEEP if c in chunk.columns]
    chunk = chunk[cols_available].copy()
    
    # Convert Indian comma numerics
    for col in ACCOUNT_NUMERIC_COLS:
        if col in chunk.columns:
            chunk[col] = indian_comma_to_numeric(chunk[col])
    
    # Parse installment amount (has '/Monthly' suffix)
    if ' INSTALLMENT-AMT' in chunk.columns:
        chunk[' INSTALLMENT-AMT'] = parse_installment_amt(chunk[' INSTALLMENT-AMT'])
    
    # Parse dates
    for col in ACCOUNT_DATE_COLS:
        if col in chunk.columns:
            chunk[col] = pd.to_datetime(chunk[col], format='%d-%m-%Y', errors='coerce')
    
    # Parse TENURE
    if 'TENURE ' in chunk.columns:
        chunk['TENURE '] = pd.to_numeric(chunk['TENURE '].astype(str).str.strip(), errors='coerce')
    
    # --- DPD History Features ---
    if 'DPD - HIST' in chunk.columns:
        dpd_parsed = chunk['DPD - HIST'].apply(parse_dpd_history)
        
        # Max DPD ever
        chunk['dpd_max_ever'] = dpd_parsed.apply(lambda x: np.nanmax(x) if len(x) > 0 and not all(np.isnan(v) for v in x if isinstance(v, float)) else np.nan)
        
        # Max DPD in last 6 months (first 6 groups)
        chunk['dpd_max_6m'] = dpd_parsed.apply(lambda x: np.nanmax(x[:6]) if len(x) >= 1 else np.nan)
        
        # Max DPD in last 12 months
        chunk['dpd_max_12m'] = dpd_parsed.apply(lambda x: np.nanmax(x[:12]) if len(x) >= 1 else np.nan)
        
        # Count months with DPD > 0 (ever)
        chunk['dpd_months_gt0'] = dpd_parsed.apply(lambda x: sum(1 for v in x if not np.isnan(v) and v > 0) if len(x) > 0 else 0)
        
        # Count months with DPD > 90 (ever)
        chunk['dpd_months_gt90'] = dpd_parsed.apply(lambda x: sum(1 for v in x if not np.isnan(v) and v > 90) if len(x) > 0 else 0)
        
        # Number of months of DPD history available
        chunk['dpd_history_length'] = dpd_parsed.apply(len)
        
        # Drop raw DPD string
        chunk.drop(columns=['DPD - HIST'], inplace=True)
    
    # Drop raw REPORTED DATE - HIST (very large, dates already captured)
    if 'REPORTED DATE - HIST' in chunk.columns:
        chunk.drop(columns=['REPORTED DATE - HIST'], inplace=True)
    
    # Clean SELF-INDICATOR to boolean
    if 'SELF-INDICATOR' in chunk.columns:
        chunk['SELF-INDICATOR'] = chunk['SELF-INDICATOR'].astype(str).str.strip().str.lower().map({'true': True, 'false': False})
    
    return chunk

print('process_account_chunk defined.')

process_account_chunk defined.


In [12]:
# QC process_account_chunk with small sample
sample = load_bureau_csv(os.path.join(SET1_DIR, 'Account.csv'), nrows=100)
sample_processed = process_account_chunk(sample)

print(f'Sample shape: {sample_processed.shape}')
print(f'Columns: {list(sample_processed.columns)}')
print(f'\nDPD features sample:')
print(sample_processed[['LOS-APP-ID', 'ACCT-TYPE', 'ACCOUNT-STATUS', 'dpd_max_ever', 'dpd_max_6m', 'dpd_months_gt0', 'dpd_history_length']].head(5))
print(f'\nNumeric cols dtypes:')
for col in ['CREDIT-LIMIT/SANC AMT', ' CURRENT-BAL', ' OVERDUE-AMT', ' INSTALLMENT-AMT']:
    if col in sample_processed.columns:
        print(f'  {col}: {sample_processed[col].dtype}, nulls={sample_processed[col].isna().sum()}')

Sample shape: (100, 34)
Columns: ['LOS-APP-ID', 'CANDIDATE - ID', 'SELF-INDICATOR', 'MATCH-TYPE', 'ACC-NUM', 'CREDIT-GRANTOR', 'ACCT-TYPE', 'CONTRIBUTOR-TYPE', 'DATE-REPORTED', 'OWNERSHIP-IND', 'ACCOUNT-STATUS', 'DISBURSED-DT', 'CLOSE-DT', 'LAST-PAYMENT-DATE', 'CREDIT-LIMIT/SANC AMT', ' DISBURSED-AMT/HIGH CREDIT', ' INSTALLMENT-AMT', ' CURRENT-BAL', 'INSTALLMENT-FREQUENCY', ' OVERDUE-AMT', ' WRITE-OFF-AMT', 'ASSET_CLASS', ' ACCOUNT-REMARKS', 'WRITE-OFF-DATE', 'INCOME ', ' INCOME INDICATOR ', 'TENURE ', ' OCCUPATION', 'dpd_max_ever', 'dpd_max_6m', 'dpd_max_12m', 'dpd_months_gt0', 'dpd_months_gt90', 'dpd_history_length']

DPD features sample:
                              LOS-APP-ID                ACCT-TYPE  \
0  FSPYLALONS000005273350_O02_5960284496         Two-Wheeler Loan   
1  FSPYLALONS000005273350_O02_5960284496  Business Loan - Secured   
2  FSTNLALONS000005227196_O02_5960217108            Property Loan   
3  FSMPLALONS000005590258_O03_5960403921            Property Loan   
4  FSM

In [13]:
full_sample = load_bureau_csv(os.path.join(SET1_DIR, 'Account.csv'))

In [14]:
full_sample['MATCH-TYPE'].value_counts()

MATCH-TYPE
PRIMARY    10374721
Name: count, dtype: int64

In [15]:
full_sample['CREDIT-LIMIT/SANC AMT'].value_counts()

CREDIT-LIMIT/SANC AMT
30,000      225712
40,000      180981
50,000      148980
15,000      144011
20,000      125017
             ...  
16,396           1
34,381           1
57,258           1
1,39,200         1
717              1
Name: count, Length: 19003, dtype: int64

In [16]:
full_sample.columns

Index(['CREDT-RPT-ID', 'LOS-APP-ID', 'CANDIDATE - ID', 'CUSTOMER ID/MBR ID',
       'BRANCH', 'KENDRA', 'SELF-INDICATOR', 'MATCH-TYPE', 'ACC-NUM',
       'CREDIT-GRANTOR', 'ACCT-TYPE', 'CONTRIBUTOR-TYPE', 'DATE-REPORTED',
       'OWNERSHIP-IND', 'ACCOUNT-STATUS', 'DISBURSED-DT', 'CLOSE-DT',
       'LAST-PAYMENT-DATE', 'CREDIT-LIMIT/SANC AMT',
       ' DISBURSED-AMT/HIGH CREDIT', ' INSTALLMENT-AMT', ' CURRENT-BAL',
       'INSTALLMENT-FREQUENCY', 'WRITE-OFF-DATE', ' OVERDUE-AMT',
       ' WRITE-OFF-AMT', 'ASSET_CLASS', ' ACCOUNT-REMARKS', 'LINKED-ACCOUNTS',
       'REPORTED DATE - HIST ', 'DPD - HIST', 'ASSET CLASS - HIST ',
       'HIGH CRD - HIST ', 'CUR BAL - HIST ', 'DAS - HIST ',
       'AMT OVERDUE - HIST ', 'AMT PAID - HIST ', 'INCOME ',
       ' INCOME INDICATOR ', 'TENURE ', ' OCCUPATION', 'Unnamed: 41'],
      dtype='str')

In [17]:
full_sample[' ACCOUNT-REMARKS'].value_counts()

 ACCOUNT-REMARKS
Written-off                             93919
Suit filed                              52075
Restructured due to COVID-19            36612
Settled                                 30753
Account Sold                            11786
Account Purchased                        6514
Post (WO) Settled                        6009
Account Purchased and Written Off        4770
Written Off and Account Sold             3384
Wilful default                           2822
Restructured Loan                        1564
Suit filed (Wilful default)               579
Restructured due to Natural Calamity      113
Restructured Loan (Govt. Mandated)         99
Restructured  and  Closed                  68
Post Write Off Closed                      27
Account Purchased and Settled              14
Account Purchased and Restructured          3
Auctioned  and  Settled                     1
Name: count, dtype: int64

In [18]:
full_sample['LINKED-ACCOUNTS'].value_counts()

LINKED-ACCOUNTS
    10374721
Name: count, dtype: int64

In [19]:
sample['LOS-APP-ID'].value_counts()

LOS-APP-ID
FSPYLALONS000005530861_O01_4872328938    23
FSTNLALONS000005279907_O02_5960277820    22
FSMPLALONS000005760230_O01_6218748577    21
FSTNLALONS000005282098_O01_3522786380    10
FSTNLALONS000005399627_O01_4218785305    10
FSPYLALONS000005273350_O02_5960284496     2
FSMPLALONS000005656630_O02_5960114594     2
FSTNLALONS000005263288_O02_5959822553     2
FSMPLALONS000005631110_O02_5960618685     2
FSTNLALONS000005227196_O02_5960217108     1
FSMPLALONS000005590258_O03_5960403921     1
FSTNLALONS000005272271_O02_5960094819     1
FSTNLALONS000005327883_O01_3887236305     1
FSTNLALONS000005108270_O02_5960035508     1
FSTNLALONS000005186685_O01_2955220070     1
Name: count, dtype: int64

In [23]:
pd.set_option('display.max_colwidth', None)
sample[sample['LOS-APP-ID'] == 'FSTNLALONS000005399627_O01_4218785305']

,CREDT-RPT-ID,LOS-APP-ID,CANDIDATE - ID,CUSTOMER ID/MBR ID,BRANCH,KENDRA,SELF-INDICATOR,MATCH-TYPE,ACC-NUM,CREDIT-GRANTOR,ACCT-TYPE,CONTRIBUTOR-TYPE,DATE-REPORTED,OWNERSHIP-IND,ACCOUNT-STATUS,DISBURSED-DT,CLOSE-DT,LAST-PAYMENT-DATE,CREDIT-LIMIT/SANC AMT,DISBURSED-AMT/HIGH CREDIT,INSTALLMENT-AMT,CURRENT-BAL,INSTALLMENT-FREQUENCY,WRITE-OFF-DATE,OVERDUE-AMT,WRITE-OFF-AMT,ASSET_CLASS,ACCOUNT-REMARKS,LINKED-ACCOUNTS,REPORTED DATE - HIST,DPD - HIST,ASSET CLASS - HIST,HIGH CRD - HIST,CUR BAL - HIST,DAS - HIST,AMT OVERDUE - HIST,AMT PAID - HIST,INCOME,INCOME INDICATOR,TENURE,OCCUPATION,Unnamed: 41
89,FIVE260128CR8268789027408034,FSTNLALONS000005399627_O01_4218785305,9553723442,NaN,NaN,NaN,True,PRIMARY,FSTNLALONS000005114446,FIVE STAR BUSINESS FINANCE LIMITED,Business Loan - Secured,NBF,31-05-2023,Individual,Closed,09-10-2019,15-03-2023,09-10-2019,NaN,"2,00,000","4,904/Monthly",0,F03,NaN,0,0,Standard,NaN,,"20230331,20230228,20230131,20221231,20221130,20221031,20220930,20220801,20220701,20220601,20220531,20220430,20220331,20220228,20220131,20211231,20211130,20211031,20210930,20210831,20210731,20210630,20210531,20210430,20210331,20210228,20210131,20201231,20201130,20201031,20200930,20200801,20200731,20200630,",000000000000000000000XXXXXXXXX000000000000000000000000000000000000000000000000000000000000000XXX000000,L01L01L01L01L01L01L01XXXXXXXXXL01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01XXXL01L01,"200000,200000,200000,200000,200000,200000,200000,,,,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000,,200000,200000,","0,106968,109732,112439,115093,117696,120248,,,,130983,133237,135398,138567,141694,144780,147827,150811,153692,156538,159350,162030,164230,166796,169313,171791,174241,176664,179079,178749,199003,,186395,186619,",S07S04S04S04S04S04S04XXXXXXXXXS04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04XXXS04S04,"0,0,0,0,0,0,0,,,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,,0,0,",",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,",0,NaN,NaN,NaN,NaN
90,FIVE260128CR8268789027408034,FSTNLALONS000005399627_O01_4218785305,5035878124,NaN,NaN,NaN,True,PRIMARY,FSTNLALONS000005399627,FIVE STAR BUSINESS FINANCE LIMITED,Property Loan,NBF,15-01-2026,Individual,Active,19-04-2023,NaN,16-12-2025,NaN,"3,00,000","7,355/Monthly","2,31,554",F03,NaN,0,0,Standard,NaN,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,20250531,20250430,20250331,20250228,20250131,20241231,20241130,20241031,20240930,20240831,20240731,20240630,20240531,20240430,20240331,20240229,20240131,20231231,20231130,20231031,20230930,20230831,20230731,20230630,20230531,20230430,",000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01,"300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,","231554,231712,234281,236798,239265,241683,244052,246374,248649,250878,252918,255066,257313,259174,261188,263308,265287,267188,268901,270867,272652,274400,274519,276346,278138,279896,281621,283214,284877,286409,287911,289384,290828,292844,",S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,",",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,",0,NaN,84.0,NaN,NaN
91,FIVE260128CR8268789027408034,FSTNLALONS000005399627_O01_4218785305,9904735285,NaN,NaN,NaN,True,PRIMARY,FSTNLALONS000005709231,FIVE STAR BUSINESS FINANCE LIMITED,Business Loan - Secured,NBF,15-01-2026,Individual,Active,31-05-2025,NaN,10-01-2026,NaN,"2,00,000","4,747/Monthly","1,87,869",F03,NaN,0,0,Standard,NaN,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,2

In [24]:
# %%time
# Load Account data in chunks and process (set1 ~10.4M rows, set2 ~6.7M rows)
CHUNK_SIZE = 500_000

account_chunks = []

for set_name, set_dir in [('set1', SET1_DIR), ('set2', SET2_DIR)]:
    filepath = os.path.join(set_dir, 'Account.csv')
    print(f'\nProcessing {set_name}: {filepath}')
    
    chunk_num = 0
    reader = pd.read_csv(
        filepath, sep='|', encoding='utf-8-sig', quotechar='"',
        chunksize=CHUNK_SIZE, on_bad_lines='skip', dtype=str
    )
    
    for chunk in reader:
        chunk_num += 1
        processed = process_account_chunk(chunk)
        account_chunks.append(processed)
        
        if chunk_num % 5 == 0:
            rows_so_far = sum(len(c) for c in account_chunks)
            print(f'  {set_name} chunk {chunk_num}: {rows_so_far:,} total rows processed')
    
    print(f'  {set_name} done: {chunk_num} chunks')

print(f'\nConcatenating {len(account_chunks)} chunks...')
account = pd.concat(account_chunks, ignore_index=True)
del account_chunks

print(f'Account combined: {account.shape}')
print(f'Unique LOS-APP-IDs in Account: {account["LOS-APP-ID"].nunique():,}')


Processing set1: /Users/shiva/Downloads/Five_star_customer_data/bureau_data/bureau_data1/FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set1_output/Account.csv
  set1 chunk 5: 2,500,000 total rows processed
  set1 chunk 10: 5,000,000 total rows processed
  set1 chunk 15: 7,500,000 total rows processed
  set1 chunk 20: 10,000,000 total rows processed
  set1 done: 21 chunks

Processing set2: /Users/shiva/Downloads/Five_star_customer_data/bureau_data/bureau_data1/FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set2_output/Account.csv
  set2 chunk 5: 12,874,721 total rows processed
  set2 chunk 10: 15,374,721 total rows processed
  set2 done: 14 chunks

Concatenating 35 chunks...
Account combined: (17073601, 34)
Unique LOS-APP-IDs in Account: 1,550,287


In [25]:
sample

,CREDT-RPT-ID,LOS-APP-ID,CANDIDATE - ID,CUSTOMER ID/MBR ID,BRANCH,KENDRA,SELF-INDICATOR,MATCH-TYPE,ACC-NUM,CREDIT-GRANTOR,ACCT-TYPE,CONTRIBUTOR-TYPE,DATE-REPORTED,OWNERSHIP-IND,ACCOUNT-STATUS,DISBURSED-DT,CLOSE-DT,LAST-PAYMENT-DATE,CREDIT-LIMIT/SANC AMT,DISBURSED-AMT/HIGH CREDIT,INSTALLMENT-AMT,CURRENT-BAL,INSTALLMENT-FREQUENCY,WRITE-OFF-DATE,OVERDUE-AMT,WRITE-OFF-AMT,ASSET_CLASS,ACCOUNT-REMARKS,LINKED-ACCOUNTS,REPORTED DATE - HIST,DPD - HIST,ASSET CLASS - HIST,HIGH CRD - HIST,CUR BAL - HIST,DAS - HIST,AMT OVERDUE - HIST,AMT PAID - HIST,INCOME,INCOME INDICATOR,TENURE,OCCUPATION,Unnamed: 41
0,FIVE260128CR9280589027408034,FSPYLALONS000005273350_O02_5960284496,7074923702,NaN,NaN,NaN,False,PRIMARY,XXXX,XXXX,Two-Wheeler Loan,PRB,15-01-2026,Individual,Suit Filed,29-10-2018,NaN,06-02-2019,NaN,"49,007",NaN,0,NaN,NaN,0,64511,NaN,Suit filed,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,20250531,20250430,20250331,20250228,20250131,20241231,20241130,20241031,20240930,20240831,20240731,20240630,20240531,20240430,20240331,20240229,20240131,20231231,20231130,20231031,20230930,20230831,20230731,20230630,20230531,20230430,20230331,20230201,",000000000000000000000000000000900900000000000000000000000000000000000000000000000000000000000000000000000XXX,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX,"49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,,","0,0,0,0,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,0,0,0,0,0,0,0,0,0,0,64511,0,0,0,0,0,0,0,,",S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16XXX,"0,0,0,0,0,0,0,0,0,0,64511,64511,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,64511,0,0,0,0,0,0,0,,",",,,,,,,,,,,,,,,0,,,,,,,,,,,,,,,,,,,,,",0,NaN,24.0,NaN,NaN
1,FIVE260128CR9280589027408034,FSPYLALONS000005273350_O02_5960284496,6944820695,NaN,NaN,NaN,True,PRIMARY,FSPYLALONS000005273350,FIVE STAR BUSINESS FINANCE LIMITED,Business Loan - Secured,NBF,15-01-2026,Joint,Active,04-03-2022,NaN,06-01-2026,NaN,"3,00,000","7,356/Monthly","1,88,239",F03,NaN,0,0,Standard,NaN,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,20250531,20250430,20250331,20250228,20250131,20241231,20241130,20241031,20240930,20240831,20240731,20240630,20240531,20240430,20240331,20240229,20240131,20231231,20231130,20231031,20230930,20230831,20230731,20230630,20230531,20230430,20230331,20230228,",000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01,"300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,","188239,195784,199185,202518,205784,208985,212121,215194,218206,221157,224044,226890,229672,232394,235062,237675,240237,242747,245206,247617,249980,252294,254560,256788,258910,261044,263136,265186,267195,269164,271093,272984,274880,276719,278469,280370,",S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,",",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,",0,NaN,84.0,NaN,NaN
2,FIVE260128CR4365789027408034,FSTNLALONS000005227196_O02_5960217108,8017120695,NaN,NaN,NaN,True,PRIMARY,FSTNLALONS000005227196,FIVE STAR BUSINESS FINANCE LIMITED,Property Loan,NBF,15-01-2026,Joint,Active,20-05-2021,NaN,06-01-2026,NaN,"3,10,000","7,601/Monthly","1,61,775",F03,NaN,0,0,Standard,NaN,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,20250531,20250430,20250331,20250228

In [26]:
# Rename Account columns to have acct_ prefix for clarity after join
acct_rename = {}
for col in account.columns:
    if col == 'LOS-APP-ID':
        continue
    clean_name = col.strip().lower().replace(' ', '_').replace('-', '_').replace('/', '_')
    acct_rename[col] = f'acct_{clean_name}'

account.rename(columns=acct_rename, inplace=True)

print(f'Account columns ({len(account.columns)}):')
print(list(account.columns))
print(f'\nAccount status distribution:')
print(account['acct_account_status'].value_counts().head(10))

Account columns (34):
['LOS-APP-ID', 'acct_candidate___id', 'acct_self_indicator', 'acct_match_type', 'acct_acc_num', 'acct_credit_grantor', 'acct_acct_type', 'acct_contributor_type', 'acct_date_reported', 'acct_ownership_ind', 'acct_account_status', 'acct_disbursed_dt', 'acct_close_dt', 'acct_last_payment_date', 'acct_credit_limit_sanc_amt', 'acct_disbursed_amt_high_credit', 'acct_installment_amt', 'acct_current_bal', 'acct_installment_frequency', 'acct_overdue_amt', 'acct_write_off_amt', 'acct_asset_class', 'acct_account_remarks', 'acct_write_off_date', 'acct_income', 'acct_income_indicator', 'acct_tenure', 'acct_occupation', 'acct_dpd_max_ever', 'acct_dpd_max_6m', 'acct_dpd_max_12m', 'acct_dpd_months_gt0', 'acct_dpd_months_gt90', 'acct_dpd_history_length']

Account status distribution:
acct_account_status
Closed                   12048044
Active                    3386595
Delinquent                1035138
Written Off                421021
Settled                     73604
Suit Filed

In [35]:
import os
# ============================================================                                                                                                                                                     
  # Read ALL Account Data from Both Sets into a Single DataFrame
  # ============================================================

account_files = [
  os.path.join(SET1_DIR, 'Account.csv'),
  os.path.join(SET2_DIR, 'Account.csv'),
]

print('Account data files to read:')
for f in account_files:
  file_size = os.path.getsize(f) / (1024**3) if os.path.exists(f) else 0
  print(f'  {f}')
  print(f'    Exists: {os.path.exists(f)}, Size: {file_size:.2f} GB')

CHUNK_SIZE = 500_000

def read_all_account_files(file_list, chunk_size):
  """Read multiple Account CSVs and return a single combined DataFrame."""
  all_chunks = []
  for filepath in file_list:
      set_label = os.path.basename(os.path.dirname(filepath))
      print(f'\nReading: {set_label}/Account.csv ...')
      reader = pd.read_csv(
          filepath, sep='|', encoding='utf-8-sig', quotechar='"',
          chunksize=chunk_size, on_bad_lines='skip', dtype=str
      )
      file_rows = 0
      chunk_count = 0
      for chunk in reader:
          all_chunks.append(chunk)
          file_rows += len(chunk)
          chunk_count += 1
          if chunk_count % 5 == 0:
              print(f'    ... {file_rows:,} rows read so far')
      print(f'  Total rows from {set_label}: {file_rows:,}')
  print(f'\nConcatenating {len(all_chunks)} chunks ...')
  combined = pd.concat(all_chunks, ignore_index=True)
  combined = combined.loc[:, ~(combined.columns.str.strip() == '')]
  return combined

account_all = read_all_account_files(account_files, CHUNK_SIZE)

print(f'\nCombined Account DataFrame Ready')
print(f'  Shape          : {account_all.shape[0]:,} rows x {account_all.shape[1]} columns')
print(f'  Unique LOS-APP-IDs : {account_all["LOS-APP-ID"].nunique():,}')
print(f'  Memory usage   : {account_all.memory_usage(deep=True).sum() / (1024**3):.2f} GB')
print(f'\nSample rows:')
account_all.head(3)


Account data files to read:
  /Users/shiva/Downloads/Five_star_customer_data/bureau_data/bureau_data1/FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set1_output/Account.csv
    Exists: True, Size: 7.23 GB
  /Users/shiva/Downloads/Five_star_customer_data/bureau_data/bureau_data1/FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set2_output/Account.csv
    Exists: True, Size: 4.46 GB

Reading: FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set1_output/Account.csv ...
    ... 2,500,000 rows read so far
    ... 5,000,000 rows read so far
    ... 7,500,000 rows read so far
    ... 10,000,000 rows read so far
  Total rows from FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set1_output: 10,374,721

Reading: FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set2_output/Account.csv ...
    ... 2,500,000 rows read so far
    ... 5,000,000 rows read so far
  Total rows from FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set2_output: 6,698,880

Concatenating 35 chunks ...

Combined Account DataFrame Ready
  Shape          : 17,073,601 rows x 42 columns
  Unique LOS-APP-IDs : 1,550,287


,CREDT-RPT-ID,LOS-APP-ID,CANDIDATE - ID,CUSTOMER ID/MBR ID,BRANCH,KENDRA,SELF-INDICATOR,MATCH-TYPE,ACC-NUM,CREDIT-GRANTOR,ACCT-TYPE,CONTRIBUTOR-TYPE,DATE-REPORTED,OWNERSHIP-IND,ACCOUNT-STATUS,DISBURSED-DT,CLOSE-DT,LAST-PAYMENT-DATE,CREDIT-LIMIT/SANC AMT,DISBURSED-AMT/HIGH CREDIT,INSTALLMENT-AMT,CURRENT-BAL,INSTALLMENT-FREQUENCY,WRITE-OFF-DATE,OVERDUE-AMT,WRITE-OFF-AMT,ASSET_CLASS,ACCOUNT-REMARKS,LINKED-ACCOUNTS,REPORTED DATE - HIST,DPD - HIST,ASSET CLASS - HIST,HIGH CRD - HIST,CUR BAL - HIST,DAS - HIST,AMT OVERDUE - HIST,AMT PAID - HIST,INCOME,INCOME INDICATOR,TENURE,OCCUPATION,Unnamed: 41
0,FIVE260128CR9280589027408034,FSPYLALONS000005273350_O02_5960284496,7074923702,NaN,NaN,NaN,false,PRIMARY,XXXX,XXXX,Two-Wheeler Loan,PRB,15-01-2026,Individual,Suit Filed,29-10-2018,NaN,06-02-2019,NaN,"49,007",NaN,0,NaN,NaN,0,64511,NaN,Suit filed,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,20250531,20250430,20250331,20250228,20250131,20241231,20241130,20241031,20240930,20240831,20240731,20240630,20240531,20240430,20240331,20240229,20240131,20231231,20231130,20231031,20230930,20230831,20230731,20230630,20230531,20230430,20230331,20230201,",000000000000000000000000000000900900000000000000000000000000000000000000000000000000000000000000000000000XXX,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX,"49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,,","0,0,0,0,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,0,0,0,0,0,0,0,0,0,0,64511,0,0,0,0,0,0,0,,",S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16XXX,"0,0,0,0,0,0,0,0,0,0,64511,64511,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,64511,0,0,0,0,0,0,0,,",",,,,,,,,,,,,,,,0,,,,,,,,,,,,,,,,,,,,,",0,NaN,24,NaN,NaN
1,FIVE260128CR9280589027408034,FSPYLALONS000005273350_O02_5960284496,6944820695,NaN,NaN,NaN,true,PRIMARY,FSPYLALONS000005273350,FIVE STAR BUSINESS FINANCE LIMITED,Business Loan - Secured,NBF,15-01-2026,Joint,Active,04-03-2022,NaN,06-01-2026,NaN,"3,00,000","7,356/Monthly","1,88,239",F03,NaN,0,0,Standard,NaN,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,20250531,20250430,20250331,20250228,20250131,20241231,20241130,20241031,20240930,20240831,20240731,20240630,20240531,20240430,20240331,20240229,20240131,20231231,20231130,20231031,20230930,20230831,20230731,20230630,20230531,20230430,20230331,20230228,",000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01,"300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,","188239,195784,199185,202518,205784,208985,212121,215194,218206,221157,224044,226890,229672,232394,235062,237675,240237,242747,245206,247617,249980,252294,254560,256788,258910,261044,263136,265186,267195,269164,271093,272984,274880,276719,278469,280370,",S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,",",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,",0,NaN,84,NaN,NaN
2,FIVE260128CR4365789027408034,FSTNLALONS000005227196_O02_5960217108,8017120695,NaN,NaN,NaN,true,PRIMARY,FSTNLALONS000005227196,FIVE STAR BUSINESS FINANCE LIMITED,Property Loan,NBF,15-01-2026,Joint,Active,20-05-2021,NaN,06-01-2026,NaN,"3,10,000","7,601/Monthly","1,61,775",F03,NaN,0,0,Standard,NaN,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,20250531,20250430,20250331,20250228,202

In [29]:
account_all

NameError: name 'account_all' is not defined

## Section 3: IOI — Inquiry Aggregation per LOS-APP-ID

In [51]:
# Load IOI (Inquiry of Inquiries) - set1 + set2
ioi1 = load_bureau_csv(os.path.join(SET1_DIR, 'IOI.csv'))
ioi2 = load_bureau_csv(os.path.join(SET2_DIR, 'IOI.csv'))

print(f'IOI set1: {ioi1.shape}, set2: {ioi2.shape}')

ioi = pd.concat([ioi1, ioi2], ignore_index=True)
del ioi1, ioi2

# Drop trailing empty column
ioi = ioi.loc[:, ~(ioi.columns.str.strip() == '')]

print(f'IOI combined: {ioi.shape}')
print(f'Columns: {list(ioi.columns)}')
ioi.head(2)

IOI set1: (3102538, 10), set2: (2154289, 10)
IOI combined: (5256827, 10)
Columns: ['CREDT-RPT-ID', 'LOS-APP-ID', 'CUSTOMER ID/MBR ID', 'CREDIT-GRANTOR', 'INQUIRY-DATE', 'OWNERSHIP-TYPE', 'PURPOSE', 'AMOUNT', 'REMARKS', 'Unnamed: 9']


,CREDT-RPT-ID,LOS-APP-ID,CUSTOMER ID/MBR ID,CREDIT-GRANTOR,INQUIRY-DATE,OWNERSHIP-TYPE,PURPOSE,AMOUNT,REMARKS,Unnamed: 9
0,FIVE260128CR4985299027408034,FSTNLALONS000005338015_O02_5959837387,NaN,XXXX,30-03-2024,PRIMARY,JLG Group,"49,000",NaN,NaN
1,FIVE260128CR4985299027408034,FSTNLALONS000005338015_O02_5959837387,NaN,XXXX,27-08-2024,PRIMARY,Microfinance Personal Loan,"75,000",NaN,NaN


In [56]:
ioi['LOS-APP-ID'].value_counts().sample()

LOS-APP-ID
FSKALALONS000005500693_O02_5960408084    5
Name: count, dtype: int64

In [58]:
ioi[ioi['LOS-APP-ID'] == 'FSKALALONS000005500693_O02_5960408084']

,CREDT-RPT-ID,LOS-APP-ID,CUSTOMER ID/MBR ID,CREDIT-GRANTOR,INQUIRY-DATE,OWNERSHIP-TYPE,PURPOSE,AMOUNT,REMARKS,Unnamed: 9
1885110,FIVE260128CR0370949027408034,FSKALALONS000005500693_O02_5960408084,NaN,XXXX,23-02-2024,PRIMARY,OTHER,0,NaN,NaN
1885111,FIVE260128CR0370949027408034,FSKALALONS000005500693_O02_5960408084,NaN,XXXX,26-02-2024,PRIMARY,OTHER,0,NaN,NaN
1885112,FIVE260128CR0370949027408034,FSKALALONS000005500693_O02_5960408084,NaN,XXXX,11-03-2024,PRIMARY,OTHER,0,NaN,NaN
1885113,FIVE260128CR0370949027408034,FSKALALONS000005500693_O02_5960408084,NaN,XXXX,26-08-2024,PRIMARY,Microfinance Personal Loan,"75,000",NaN,NaN
1885114,FIVE260128CR0370949027408034,FSKALALONS000005500693_O02_5960408084,NaN,XXXX,18-07-2025,PRIMARY,Microfinance Personal Loan,"1,50,000",NaN,NaN


In [59]:
ioi['OWNERSHIP-TYPE'].value_counts()

OWNERSHIP-TYPE
PRIMARY    5256827
Name: count, dtype: int64

In [64]:
ioi['Unnamed: 9'].value_counts().head(20).reset_index()

,Unnamed: 9,count


In [62]:
ioi['PURPOSE'].value_counts().head(20).reset_index()

,PURPOSE,count
0,OTHER,1880160
1,Other,1145488
2,Microfinance Personal Loan,348135
3,JLG Individual,313720
4,Microfinance Business Loan,280984
5,Personal Loan,233473
6,JLG Group,164640
7,Housing Loan,143990
8,Credit Card,119458
9,Consumer Loan,117670


In [31]:
# Parse IOI fields
ioi['INQUIRY-DATE'] = pd.to_datetime(ioi['INQUIRY-DATE'], format='%d-%m-%Y', errors='coerce')
ioi['AMOUNT'] = indian_comma_to_numeric(ioi['AMOUNT'])

# Reference date for recency calculations
ref_date = pd.Timestamp('2025-09-01')  # approximate bureau pull date

# Flag: inquiry within last 6 months / 12 months
ioi['inq_within_6m'] = ((ref_date - ioi['INQUIRY-DATE']).dt.days <= 180).astype(int)
ioi['inq_within_12m'] = ((ref_date - ioi['INQUIRY-DATE']).dt.days <= 365).astype(int)

# Aggregate per LOS-APP-ID
ioi_agg = ioi.groupby('LOS-APP-ID').agg(
    ioi_total_inquiries=('INQUIRY-DATE', 'count'),
    ioi_inquiries_6m=('inq_within_6m', 'sum'),
    ioi_inquiries_12m=('inq_within_12m', 'sum'),
    ioi_unique_grantors=('CREDIT-GRANTOR', 'nunique'),
    ioi_unique_purposes=('PURPOSE', 'nunique'),
    ioi_amount_mean=('AMOUNT', 'mean'),
    ioi_amount_max=('AMOUNT', 'max'),
    ioi_amount_sum=('AMOUNT', 'sum'),
    ioi_latest_inquiry=('INQUIRY-DATE', 'max'),
    ioi_earliest_inquiry=('INQUIRY-DATE', 'min'),
).reset_index()

del ioi

print(f'IOI aggregated: {ioi_agg.shape}')
print(f'Unique LOS-APP-IDs: {ioi_agg["LOS-APP-ID"].nunique():,}')
ioi_agg.head(2)

IOI aggregated: (1163520, 11)
Unique LOS-APP-IDs: 1,163,520


,LOS-APP-ID,ioi_total_inquiries,ioi_inquiries_6m,ioi_inquiries_12m,ioi_unique_grantors,ioi_unique_purposes,ioi_amount_mean,ioi_amount_max,ioi_amount_sum,ioi_latest_inquiry,ioi_earliest_inquiry
0,100007_O01_341666128,4,2,3,1,2,15000.0,15000,60000,2025-11-27,2024-02-26
1,100009_O01_341666137,1,0,1,1,1,0.0,0,0,2025-03-03,2025-03-03


## Section 4: Inquiry — Applicant Demographics

In [66]:
# Load Inquiry (demographics) - set1 + set2
inquiry1 = load_bureau_csv(os.path.join(SET1_DIR, 'Inquiry.csv'))
inquiry2 = load_bureau_csv(os.path.join(SET2_DIR, 'Inquiry.csv'))

print(f'Inquiry set1: {inquiry1.shape}, set2: {inquiry2.shape}')

inquiry = pd.concat([inquiry1, inquiry2], ignore_index=True)
del inquiry1, inquiry2

# Drop trailing empty column
inquiry = inquiry.loc[:, ~(inquiry.columns.str.strip() == '')]

print(f'Inquiry combined: {inquiry.shape}')
print(f'Duplicates on LOS-APP-ID: {inquiry["LOS-APP-ID"].duplicated().sum()}')

Inquiry set1: (999998, 32), set2: (550373, 32)
Inquiry combined: (1550371, 32)
Duplicates on LOS-APP-ID: 0


In [74]:
inquiry.head()

,CREDT-RPT-ID,LOS-APP-ID,BRANCH-ID,CUSTOMER ID/MBR ID,APPLICANT-NAME,AKA,DOB,PAN,PASSPORT,DL,RC,UID,VOTERS,OTHER-ID,FATHER,SPOUSE,MOTHER,OTHER,PHONE 1,PHONE 2,PHONE 3,EMAIL 1,EMAIL 2,GENDER,ADDRESS-1,STATE-1,ADDRESS-2,STATE-2,ADDRESS-3,STATE-3,RETRO-PR-DT,Unnamed: 31
0,FIVE260128CR6620529027408034,FSAPLALONS000005541370_O01_4940150518,NaN,NaN,B RAMAKRISHNA,NaN,01-01-1971,NaN,NaN,NaN,NaN,NaN,TVF1779552,NaN,NaN,NaN,NaN,NaN,9.502494e+09,NaN,NaN,NaN,NaN,Male,"GOLLAPALLI, SOMALA MANDALAM, NELLIMANDA, , CHITTOOR, 517257 AP",AP,NaN,NaN,NaN,NaN,NaN,NaN
1,FIVE260128CR0910029027408034,FSAPLALONS000005490062_O01_4691903684,NaN,NaN,SIVVADA VENKATESH,NaN,01-01-1966,CRZPV4731R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.305333e+09,NaN,NaN,NaN,NaN,Male,"D NO 1/160 BAIRAJU KANDRIGA PAVANIVARI KANDRIGA K V B PURAM, D NO 1/160 BAIRAJU KANDRIGA PAVANIVARI KANDRIGA K V B PURAM, , CHITTOOR, 517643 AP",AP,NaN,NaN,NaN,NaN,NaN,NaN
2,FIVE260128CR7125729027408034,FSAPLALONS000005564548_O02_5960904981,NaN,NaN,MOLLA MAHABOOB BASHA,NaN,08-06-1972,DZZPM4361K,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.100387e+09,NaN,NaN,NaN,NaN,Male,"MUSUM STREET CHERKUCHERLA BYRAPURAM KURNOOLKURNOOL, KURNOOL, 518405 AP",AP,NaN,NaN,NaN,NaN,NaN,NaN
3,FIVE260128CR4915229027408034,FSAPLALONS000005448452_O01_4435969439,NaN,NaN,KOMMU HANUMANTHU,NaN,15-09-2004,BTTPH8338D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.780302e+09,NaN,NaN,NaN,NaN,Male,"H.NO.4-14-A, VELDURTHI, KURNOOL, KURNOOL 518216 AP",AP,NaN,NaN,NaN,NaN,NaN,NaN
4,FIVE260128CR6131529027408034,FSAPLALONS000005468308_O03_5960544769,NaN,NaN,PALUGUNDLA ADHINARAYANA,NaN,01-01-1955,NaN,NaN,NaN,NaN,NaN,RLY1555473,NaN,NaN,NaN,NaN,NaN,8.498057e+09,NaN,NaN,NaN,NaN,Male,"101 MAIN ROAD MAMPUDI MANDALAM ANKEPALLI PRAKASAM, PRAKASAM, 523240 AP",AP,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
inquiry['BRANCH-ID'].value_counts()

Series([], Name: count, dtype: int64)

In [76]:
inquiry['CUSTOMER ID/MBR ID'].value_counts()

Series([], Name: count, dtype: int64)

In [92]:
inquiry['EMAIL 2'].value_counts()

Series([], Name: count, dtype: int64)

In [94]:
inquiry['STATE-2'].value_counts()

Series([], Name: count, dtype: int64)

In [96]:
inquiry['RETRO-PR-DT'].value_counts()

Series([], Name: count, dtype: int64)

In [97]:
# Extract useful demographic fields
inquiry_cols = ['LOS-APP-ID']

# DOB -> age
if 'DOB' in inquiry.columns:
    inquiry['DOB'] = pd.to_datetime(inquiry['DOB'], format='%d-%m-%Y', errors='coerce')
    inquiry['demog_age'] = ((ref_date - inquiry['DOB']).dt.days / 365.25).round(1)
    inquiry_cols.append('demog_age')

# Gender
if 'GENDER' in inquiry.columns:
    inquiry['demog_gender'] = inquiry['GENDER'].str.strip()
    inquiry_cols.append('demog_gender')

# PAN presence
if 'PAN' in inquiry.columns:
    inquiry['demog_has_pan'] = inquiry['PAN'].notna() & (inquiry['PAN'].str.strip() != '')
    inquiry_cols.append('demog_has_pan')

# State
if 'STATE-1' in inquiry.columns:
    inquiry['demog_state'] = inquiry['STATE-1'].str.strip()
    inquiry_cols.append('demog_state')

inquiry_demog = inquiry[inquiry_cols].copy()
del inquiry

print(f'Inquiry demographics: {inquiry_demog.shape}')
inquiry_demog.head(3)

Inquiry demographics: (1550371, 5)


,LOS-APP-ID,demog_age,demog_gender,demog_has_pan,demog_state
0,FSAPLALONS000005541370_O01_4940150518,55.0,Male,False,AP
1,FSAPLALONS000005490062_O01_4691903684,60.0,Male,True,AP
2,FSAPLALONS000005564548_O02_5960904981,53.6,Male,True,AP


## Section 5: Demog — Demographic Element Counts (No PII Values)

In [98]:
# Load only LOS-APP-ID and ELEMENT-TYPE (skip PII values)
demog1 = load_bureau_csv(os.path.join(SET1_DIR, 'Demog.csv'), usecols=['LOS-APP-ID', 'ELEMENT-TYPE'])
demog2 = load_bureau_csv(os.path.join(SET2_DIR, 'Demog.csv'), usecols=['LOS-APP-ID', 'ELEMENT-TYPE'])

print(f'Demog set1: {demog1.shape}, set2: {demog2.shape}')

demog = pd.concat([demog1, demog2], ignore_index=True)
del demog1, demog2

print(f'Demog combined: {demog.shape}')
print(f'\nELEMENT-TYPE distribution:')
print(demog['ELEMENT-TYPE'].value_counts())

Demog set1: (7836182, 2), set2: (4508703, 2)
Demog combined: (12344885, 2)

ELEMENT-TYPE distribution:
ELEMENT-TYPE
GENDER        1550287
ADDRESS-1     1549954
PHONE-1       1549832
VOTERID       1148093
ADDRESS-2     1136906
ADDRESS-3      982391
PAN            981365
PHONE-2        916256
UID            875449
PHONE-3        592681
EMAIL-ID-1     391187
OTHERID        290794
EMAIL-ID-2     155618
RATIONCARD     149188
EMAIL-ID-3      64546
PASSPORTNO      10338
Name: count, dtype: int64


In [100]:
demog.sample(10)

,LOS-APP-ID,ELEMENT-TYPE
12228828,FSTNLALONS000005714343_O01_5858516665,ADDRESS-1
2574943,FSAPLALONS000005745703_O03_6094193967,VOTERID
5168953,FSMPLALONS000005644860_O02_5960626359,EMAIL-ID-1
8736379,FSTSLALONS000005438645_O02_5960492916,PHONE-1
1313759,FSAPLALONS000005381114_O01_4167350721,ADDRESS-2
4302380,FSMPLALONS000005595578_O03_5960693338,VOTERID
6987755,FSAPLALONS000005652561_O01_5518467966,VOTERID
3713732,FSAPLALONS000005012089_O01_2041236091,UID
3442866,FSMPLALONS000005767708_O01_6286475069,ADDRESS-1
11986404,FSTSLALONS000005645889_O01_5421741386,GENDER


In [16]:
# Pivot: count of each element type per LOS-APP-ID
demog['val'] = 1
demog_pivot = demog.pivot_table(
    index='LOS-APP-ID', columns='ELEMENT-TYPE', values='val',
    aggfunc='sum', fill_value=0
).reset_index()

del demog

# Rename columns with demog_ prefix
demog_rename = {col: f'demog_count_{col.lower().replace("-", "_").replace(" ", "_")}' 
                for col in demog_pivot.columns if col != 'LOS-APP-ID'}
demog_pivot.rename(columns=demog_rename, inplace=True)

print(f'Demog pivoted: {demog_pivot.shape}')
print(f'Columns: {list(demog_pivot.columns)}')
demog_pivot.head(2)

Demog pivoted: (1550287, 17)
Columns: ['LOS-APP-ID', 'demog_count_address_1', 'demog_count_address_2', 'demog_count_address_3', 'demog_count_email_id_1', 'demog_count_email_id_2', 'demog_count_email_id_3', 'demog_count_gender', 'demog_count_otherid', 'demog_count_pan', 'demog_count_passportno', 'demog_count_phone_1', 'demog_count_phone_2', 'demog_count_phone_3', 'demog_count_rationcard', 'demog_count_uid', 'demog_count_voterid']


ELEMENT-TYPE,LOS-APP-ID,demog_count_address_1,demog_count_address_2,demog_count_address_3,demog_count_email_id_1,demog_count_email_id_2,demog_count_email_id_3,demog_count_gender,demog_count_otherid,demog_count_pan,demog_count_passportno,demog_count_phone_1,demog_count_phone_2,demog_count_phone_3,demog_count_rationcard,demog_count_uid,demog_count_voterid
0,100007_O01_341666128,1,1,1,1,1,1,1,0,1,0,1,1,1,0,1,1
1,100008_O01_341666131,1,1,1,0,0,0,1,0,1,0,1,1,1,1,0,0


## Section 6: Master Join

In [5]:
import os
# ============================================================                                                                                                                                                     
  # Read ALL Account Data from Both Sets into a Single DataFrame
  # ============================================================

account_files = [
  os.path.join(SET1_DIR, 'Account.csv'),
  os.path.join(SET2_DIR, 'Account.csv'),
]

print('Account data files to read:')
for f in account_files:
  file_size = os.path.getsize(f) / (1024**3) if os.path.exists(f) else 0
  print(f'  {f}')
  print(f'    Exists: {os.path.exists(f)}, Size: {file_size:.2f} GB')

CHUNK_SIZE = 500_000

def read_all_account_files(file_list, chunk_size):
  """Read multiple Account CSVs and return a single combined DataFrame."""
  all_chunks = []
  for filepath in file_list:
      set_label = os.path.basename(os.path.dirname(filepath))
      print(f'\nReading: {set_label}/Account.csv ...')
      reader = pd.read_csv(
          filepath, sep='|', encoding='utf-8-sig', quotechar='"',
          chunksize=chunk_size, on_bad_lines='skip', dtype=str
      )
      file_rows = 0
      chunk_count = 0
      for chunk in reader:
          all_chunks.append(chunk)
          file_rows += len(chunk)
          chunk_count += 1
          if chunk_count % 5 == 0:
              print(f'    ... {file_rows:,} rows read so far')
      print(f'  Total rows from {set_label}: {file_rows:,}')
  print(f'\nConcatenating {len(all_chunks)} chunks ...')
  combined = pd.concat(all_chunks, ignore_index=True)
  combined = combined.loc[:, ~(combined.columns.str.strip() == '')]
  return combined

account_all = read_all_account_files(account_files, CHUNK_SIZE)

print(f'\nCombined Account DataFrame Ready')
print(f'  Shape          : {account_all.shape[0]:,} rows x {account_all.shape[1]} columns')
print(f'  Unique LOS-APP-IDs : {account_all["LOS-APP-ID"].nunique():,}')
print(f'  Memory usage   : {account_all.memory_usage(deep=True).sum() / (1024**3):.2f} GB')
print(f'\nSample rows:')
account_all.head(3)

Account data files to read:
  /Users/shiva/Downloads/Five_star_customer_data/bureau_data/bureau_data1/FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set1_output/Account.csv
    Exists: True, Size: 7.23 GB
  /Users/shiva/Downloads/Five_star_customer_data/bureau_data/bureau_data1/FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set2_output/Account.csv
    Exists: True, Size: 4.46 GB

Reading: FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set1_output/Account.csv ...
    ... 2,500,000 rows read so far
    ... 5,000,000 rows read so far
    ... 7,500,000 rows read so far
    ... 10,000,000 rows read so far
  Total rows from FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set1_output: 10,374,721

Reading: FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set2_output/Account.csv ...
    ... 2,500,000 rows read so far
    ... 5,000,000 rows read so far
  Total rows from FIVESTAR_CNS_ACTIVE_ACCOUNT_DM_1_set2_output: 6,698,880

Concatenating 35 chunks ...

Combined Account DataFrame Ready
  Shape          : 17,073,601 rows x 42 columns
  Unique LOS-APP-IDs : 1,550,287


,CREDT-RPT-ID,LOS-APP-ID,CANDIDATE - ID,CUSTOMER ID/MBR ID,BRANCH,KENDRA,SELF-INDICATOR,MATCH-TYPE,ACC-NUM,CREDIT-GRANTOR,ACCT-TYPE,CONTRIBUTOR-TYPE,DATE-REPORTED,OWNERSHIP-IND,ACCOUNT-STATUS,DISBURSED-DT,CLOSE-DT,LAST-PAYMENT-DATE,CREDIT-LIMIT/SANC AMT,DISBURSED-AMT/HIGH CREDIT,INSTALLMENT-AMT,CURRENT-BAL,INSTALLMENT-FREQUENCY,WRITE-OFF-DATE,OVERDUE-AMT,WRITE-OFF-AMT,ASSET_CLASS,ACCOUNT-REMARKS,LINKED-ACCOUNTS,REPORTED DATE - HIST,DPD - HIST,ASSET CLASS - HIST,HIGH CRD - HIST,CUR BAL - HIST,DAS - HIST,AMT OVERDUE - HIST,AMT PAID - HIST,INCOME,INCOME INDICATOR,TENURE,OCCUPATION,Unnamed: 41
0,FIVE260128CR9280589027408034,FSPYLALONS000005273350_O02_5960284496,7074923702,NaN,NaN,NaN,false,PRIMARY,XXXX,XXXX,Two-Wheeler Loan,PRB,15-01-2026,Individual,Suit Filed,29-10-2018,NaN,06-02-2019,NaN,"49,007",NaN,0,NaN,NaN,0,64511,NaN,Suit filed,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,2025...",0000000000000000000000000000009009000000000000000000000000000000000000000000...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,"49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,49007,4900...","0,0,0,0,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,64511,64...",S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S16S...,"0,0,0,0,0,0,0,0,0,0,64511,64511,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,64511,0,0,0,0,...",",,,,,,,,,,,,,,,0,,,,,,,,,,,,,,,,,,,,,",0,NaN,24,NaN,NaN
1,FIVE260128CR9280589027408034,FSPYLALONS000005273350_O02_5960284496,6944820695,NaN,NaN,NaN,true,PRIMARY,FSPYLALONS000005273350,FIVE STAR BUSINESS FINANCE LIMITED,Business Loan - Secured,NBF,15-01-2026,Joint,Active,04-03-2022,NaN,06-01-2026,NaN,"3,00,000","7,356/Monthly","1,88,239",F03,NaN,0,0,Standard,NaN,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,2025...",0000000000000000000000000000000000000000000000000000000000000000000000000000...,L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L...,"300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000...","188239,195784,199185,202518,205784,208985,212121,215194,218206,221157,224044...",S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,",",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,",0,NaN,84,NaN,NaN
2,FIVE260128CR4365789027408034,FSTNLALONS000005227196_O02_5960217108,8017120695,NaN,NaN,NaN,true,PRIMARY,FSTNLALONS000005227196,FIVE STAR BUSINESS FINANCE LIMITED,Property Loan,NBF,15-01-2026,Joint,Active,20-05-2021,NaN,06-01-2026,NaN,"3,10,000","7,601/Monthly","1,61,775",F03,NaN,0,0,Standard,NaN,,"20260115,20251231,20251130,20251031,20250930,20250831,20250731,20250630,2025...",0000000000000000000000000000000000000000000000000000000000000000000000000000...,L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L01L...,"310000,310000,310000,310000,310000,310000,310000,310000,310000,310000,310000...","161775,161913,166173,170347,174438,178447,182375,186225,189998,193695,197316...",S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S04S...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,",",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,",0,NaN,84,NaN,NaN


In [13]:
fill_rate_df = calculate_fill_rates(account_all)
print(fill_rate_df.to_string(index=False))

                    Column  Total Rows  Non-Null Count  Null Count  Fill Rate (%)
              CREDT-RPT-ID    17073601        17073601           0         100.00
             OWNERSHIP-IND    17073601        17073601           0         100.00
       AMT OVERDUE - HIST     17073601        17073601           0         100.00
               DAS - HIST     17073601        17073601           0         100.00
           CUR BAL - HIST     17073601        17073601           0         100.00
                DPD - HIST    17073601        17073601           0         100.00
     REPORTED DATE - HIST     17073601        17073601           0         100.00
           LINKED-ACCOUNTS    17073601        17073601           0         100.00
             WRITE-OFF-AMT    17073601        17073601           0         100.00
                LOS-APP-ID    17073601        17073601           0         100.00
 DISBURSED-AMT/HIGH CREDIT    17073601        17073601           0         100.00
            ACCO

In [15]:
account_all.columns

Index(['CREDT-RPT-ID', 'LOS-APP-ID', 'CANDIDATE - ID', 'CUSTOMER ID/MBR ID',
       'BRANCH', 'KENDRA', 'SELF-INDICATOR', 'MATCH-TYPE', 'ACC-NUM',
       'CREDIT-GRANTOR', 'ACCT-TYPE', 'CONTRIBUTOR-TYPE', 'DATE-REPORTED',
       'OWNERSHIP-IND', 'ACCOUNT-STATUS', 'DISBURSED-DT', 'CLOSE-DT',
       'LAST-PAYMENT-DATE', 'CREDIT-LIMIT/SANC AMT',
       ' DISBURSED-AMT/HIGH CREDIT', ' INSTALLMENT-AMT', ' CURRENT-BAL',
       'INSTALLMENT-FREQUENCY', 'WRITE-OFF-DATE', ' OVERDUE-AMT',
       ' WRITE-OFF-AMT', 'ASSET_CLASS', ' ACCOUNT-REMARKS', 'LINKED-ACCOUNTS',
       'REPORTED DATE - HIST ', 'DPD - HIST', 'ASSET CLASS - HIST ',
       'HIGH CRD - HIST ', 'CUR BAL - HIST ', 'DAS - HIST ',
       'AMT OVERDUE - HIST ', 'AMT PAID - HIST ', 'INCOME ',
       ' INCOME INDICATOR ', 'TENURE ', ' OCCUPATION', 'Unnamed: 41'],
      dtype='str')

In [16]:
columns_considering = ['CREDT-RPT-ID', 'LOS-APP-ID', 'CANDIDATE - ID',
       'BRANCH', 'KENDRA', 'SELF-INDICATOR', 'MATCH-TYPE', 'ACC-NUM',
       'CREDIT-GRANTOR', 'ACCT-TYPE', 'CONTRIBUTOR-TYPE', 'DATE-REPORTED',
       'OWNERSHIP-IND', 'ACCOUNT-STATUS', 'DISBURSED-DT', 'CLOSE-DT',
       'LAST-PAYMENT-DATE', 'CREDIT-LIMIT/SANC AMT',
       ' DISBURSED-AMT/HIGH CREDIT', ' INSTALLMENT-AMT', ' CURRENT-BAL',
       'INSTALLMENT-FREQUENCY', 'WRITE-OFF-DATE', ' OVERDUE-AMT',
       ' WRITE-OFF-AMT', 'ASSET_CLASS',
       'REPORTED DATE - HIST ', 'DPD - HIST',
       'HIGH CRD - HIST ', 'CUR BAL - HIST ',
       'AMT OVERDUE - HIST ', 'AMT PAID - HIST ', 'INCOME ',
       ' INCOME INDICATOR ', 'TENURE ', ' OCCUPATION']

In [19]:
account_all = account_all[columns_considering]

In [20]:
account_all.shape

(17073601, 36)

In [38]:
# Load IOI (Inquiry of Inquiries) - set1 + set2
ioi1 = load_bureau_csv(os.path.join(SET1_DIR, 'IOI.csv'))
ioi2 = load_bureau_csv(os.path.join(SET2_DIR, 'IOI.csv'))

print(f'IOI set1: {ioi1.shape}, set2: {ioi2.shape}')

ioi = pd.concat([ioi1, ioi2], ignore_index=True)
del ioi1, ioi2

# Drop trailing empty column
ioi = ioi.loc[:, ~(ioi.columns.str.strip() == '')]

print(f'IOI combined: {ioi.shape}')
print(f'Columns: {list(ioi.columns)}')
ioi.head(2)

IOI set1: (3102538, 10), set2: (2154289, 10)
IOI combined: (5256827, 10)
Columns: ['CREDT-RPT-ID', 'LOS-APP-ID', 'CUSTOMER ID/MBR ID', 'CREDIT-GRANTOR', 'INQUIRY-DATE', 'OWNERSHIP-TYPE', 'PURPOSE', 'AMOUNT', 'REMARKS', 'Unnamed: 9']


,CREDT-RPT-ID,LOS-APP-ID,CUSTOMER ID/MBR ID,CREDIT-GRANTOR,INQUIRY-DATE,OWNERSHIP-TYPE,PURPOSE,AMOUNT,REMARKS,Unnamed: 9
0,FIVE260128CR4985299027408034,FSTNLALONS000005338015_O02_5959837387,NaN,XXXX,30-03-2024,PRIMARY,JLG Group,"49,000",NaN,NaN
1,FIVE260128CR4985299027408034,FSTNLALONS000005338015_O02_5959837387,NaN,XXXX,27-08-2024,PRIMARY,Microfinance Personal Loan,"75,000",NaN,NaN


In [42]:
ioi['CREDIT-GRANTOR'].value_counts()

CREDIT-GRANTOR
XXXX                                  4157689
FIVE STAR BUSINESS FINANCE LIMITED    1099138
Name: count, dtype: int64

In [44]:
ioi_df = ioi
# ── DIAGNOSTICS ──────────────────────────────────────────────────────────────
print("Shape:", ioi_df.shape)
print("\nColumns:", ioi_df.columns.tolist())
print("\nSample CREDIT-GRANTOR values:")
print(ioi_df["CREDIT-GRANTOR"].value_counts().head(10))
print("\nSample CUSTOMER ID values:")
print(ioi_df["CUSTOMER ID/MBR ID"].value_counts().head(5))
print("\nNull counts:")
print(ioi_df[["CUSTOMER ID/MBR ID", "CREDIT-GRANTOR", "INQUIRY-DATE", "AMOUNT"]].isnull().sum())
print("\nFirst 3 rows:")
print(ioi_df.head(3).T)

Shape: (5256827, 10)

Columns: ['CREDT-RPT-ID', 'LOS-APP-ID', 'CUSTOMER ID/MBR ID', 'CREDIT-GRANTOR', 'INQUIRY-DATE', 'OWNERSHIP-TYPE', 'PURPOSE', 'AMOUNT', 'REMARKS', 'Unnamed: 9']

Sample CREDIT-GRANTOR values:
CREDIT-GRANTOR
XXXX                                  4157689
FIVE STAR BUSINESS FINANCE LIMITED    1099138
Name: count, dtype: int64

Sample CUSTOMER ID values:
Series([], Name: count, dtype: int64)

Null counts:
CUSTOMER ID/MBR ID    5256827
CREDIT-GRANTOR              0
INQUIRY-DATE                0
AMOUNT                      0
dtype: int64

First 3 rows:
                                                        0  \
CREDT-RPT-ID                 FIVE260128CR4985299027408034   
LOS-APP-ID          FSTNLALONS000005338015_O02_5959837387   
CUSTOMER ID/MBR ID                                    NaN   
CREDIT-GRANTOR                                       XXXX   
INQUIRY-DATE                                   30-03-2024   
OWNERSHIP-TYPE                                    PRIMARY   

In [46]:
df = ioi_df.copy()
df.columns = df.columns.str.strip()

# Step 1: Check date parsing
df["INQUIRY-DATE"] = pd.to_datetime(df["INQUIRY-DATE"], dayfirst=True, errors="coerce")
print("NaT after parsing:", df["INQUIRY-DATE"].isna().sum())
print("Date range:", df["INQUIRY-DATE"].min(), "→", df["INQUIRY-DATE"].max())

# Step 2: Check how many rows survive the date filter
AS_OF_DATE = pd.Timestamp("2025-09-30")
df["DAYS_SINCE_INQUIRY"] = (AS_OF_DATE - df["INQUIRY-DATE"]).dt.days
mask = df["INQUIRY-DATE"] <= AS_OF_DATE
print("Rows before filter:", len(df))
print("Rows after filter:", mask.sum())
print("Rows dropped (future):", (~mask).sum())

# Step 3: Check groupby actually produces groups
df_filtered = df[mask]
print("\nUnique LOS-APP-IDs:", df_filtered["LOS-APP-ID"].nunique())
print("LOS-APP-ID nulls:", df_filtered["LOS-APP-ID"].isna().sum())
print("Sample LOS-APP-IDs:", df_filtered["LOS-APP-ID"].dropna().head(5).tolist())

NaT after parsing: 0
Date range: 2024-01-01 00:00:00 → 2026-01-28 00:00:00
Rows before filter: 5256827
Rows after filter: 4463704
Rows dropped (future): 793123

Unique LOS-APP-IDs: 1101444
LOS-APP-ID nulls: 0
Sample LOS-APP-IDs: ['FSTNLALONS000005338015_O02_5959837387', 'FSTNLALONS000005338015_O02_5959837387', 'FSTNLALONS000005389748_O02_5960454327', 'FSTNLALONS000005389748_O02_5960454327', 'FSTNLALONS000005345444_O02_5959825433']


In [47]:
import pandas as pd
import numpy as np

AS_OF_DATE = pd.Timestamp("2025-09-30")
FIVE_STAR = "FIVE STAR BUSINESS FINANCE LIMITED"

# ── Prep ──────────────────────────────────────────────────────────────────────
df = ioi_df.copy()
df.columns = df.columns.str.strip()
df["INQUIRY-DATE"] = pd.to_datetime(df["INQUIRY-DATE"], dayfirst=True, errors="coerce")
df["AMOUNT_CLEAN"] = df["AMOUNT"].apply(lambda x: float(str(x).replace(",", "").strip()) if pd.notna(x) else np.nan)
df["DAYS_SINCE_INQUIRY"] = (AS_OF_DATE - df["INQUIRY-DATE"]).dt.days

# Filter future dates
df = df[df["INQUIRY-DATE"] <= AS_OF_DATE].copy()

# Boolean flags
df["IS_FIVESTAR"] = df["CREDIT-GRANTOR"].str.strip().str.upper() == FIVE_STAR.upper()
df["IS_XXXX"]     = df["CREDIT-GRANTOR"].str.strip() == "XXXX"
df["IS_MFI"]      = df["PURPOSE"].str.upper().str.contains("MICROFINANCE|MFI|JLG", na=False)
df["IS_MFL"]      = df["PURPOSE"].str.upper().str.contains("MICROFINANCE PERSONAL LOAN", na=False)

# Time window flags
df["INQ_3M"]  = df["DAYS_SINCE_INQUIRY"] <= 90
df["INQ_6M"]  = df["DAYS_SINCE_INQUIRY"] <= 180
df["INQ_12M"] = df["DAYS_SINCE_INQUIRY"] <= 365

G = "LOS-APP-ID"

# ── ALL-LENDER aggregations ───────────────────────────────────────────────────
all_agg = df.groupby(G).agg(
    ioi_total_inquiries       = (G, "count"),
    ioi_inq_last_3m           = ("INQ_3M", "sum"),
    ioi_inq_last_6m           = ("INQ_6M", "sum"),
    ioi_inq_last_12m          = ("INQ_12M", "sum"),
    ioi_days_since_last_inq   = ("DAYS_SINCE_INQUIRY", "min"),
    ioi_days_since_first_inq  = ("DAYS_SINCE_INQUIRY", "max"),
    ioi_unique_lenders        = ("CREDIT-GRANTOR", "nunique"),
    ioi_mfi_inq_count         = ("IS_MFI", "sum"),
    ioi_other_lender_inq_count= ("IS_XXXX", "sum"),
    ioi_max_inq_amt           = ("AMOUNT_CLEAN", "max"),
    ioi_total_inq_amt         = ("AMOUNT_CLEAN", "sum"),
    ioi_avg_inq_amt           = ("AMOUNT_CLEAN", "mean"),
).reset_index()

# MFI inquiries in last 6m (compound flag)
all_agg["ioi_mfi_inq_last_6m"] = (
    df[df["IS_MFI"] & df["INQ_6M"]].groupby(G).size().reindex(all_agg[G], fill_value=0).values
)

# Rapid burst: min gap between inquiry dates per loan
gap_df = (
    df.sort_values([G, "INQUIRY-DATE"])
    .groupby(G)["INQUIRY-DATE"]
    .apply(lambda x: x.diff().dt.days.min() if len(x) > 1 else np.nan)
    .reset_index()
    .rename(columns={"INQUIRY-DATE": "ioi_min_gap_between_inq_days"})
)
burst_df = (
    df.sort_values([G, "INQUIRY-DATE"])
    .groupby(G)["INQUIRY-DATE"]
    .apply(lambda x: int((x.diff().dt.days <= 30).sum()) if len(x) > 1 else 0)
    .reset_index()
    .rename(columns={"INQUIRY-DATE": "ioi_rapid_inq_bursts"})
)

# ── FIVE STAR aggregations ────────────────────────────────────────────────────
fs_df = df[df["IS_FIVESTAR"]]

fs_agg = fs_df.groupby(G).agg(
    ioi_fs_inq_count          = (G, "count"),
    ioi_fs_inq_last_3m        = ("INQ_3M", "sum"),
    ioi_fs_inq_last_6m        = ("INQ_6M", "sum"),
    ioi_fs_days_since_last_inq= ("DAYS_SINCE_INQUIRY", "min"),
    ioi_fs_max_inq_amt        = ("AMOUNT_CLEAN", "max"),
    ioi_fs_total_inq_amt      = ("AMOUNT_CLEAN", "sum"),
    ioi_fs_has_jlg_inq        = ("IS_MFI", "max"),
    ioi_fs_has_mfl_inq        = ("IS_MFL", "max"),
).reset_index()

# ── OTHER LENDER (XXXX) aggregations ─────────────────────────────────────────
oth_df = df[df["IS_XXXX"]]

oth_agg = oth_df.groupby(G).agg(
    ioi_other_inq_last_6m    = ("INQ_6M", "sum"),
    ioi_other_mfi_inq_count  = ("IS_MFI", "sum"),
    ioi_other_max_inq_amt    = ("AMOUNT_CLEAN", "max"),
).reset_index()

# ── Merge all together ────────────────────────────────────────────────────────
ioi_features = (
    all_agg
    .merge(gap_df,  on=G, how="left")
    .merge(burst_df, on=G, how="left")
    .merge(fs_agg,  on=G, how="left")
    .merge(oth_agg, on=G, how="left")
)

# Fill nulls for FS/other features (loans with no FS/other inquiries)
fs_cols  = [c for c in ioi_features.columns if c.startswith("ioi_fs_")]
oth_cols = [c for c in ioi_features.columns if c.startswith("ioi_other_")]
ioi_features[fs_cols]  = ioi_features[fs_cols].fillna(0)
ioi_features[oth_cols] = ioi_features[oth_cols].fillna(0)

# Derived ratio
ioi_features["ioi_fs_inq_share"] = (
    ioi_features["ioi_fs_inq_count"] / ioi_features["ioi_total_inquiries"]
)

print("Features shape:", ioi_features.shape)
print(ioi_features.head(3).T)

Features shape: (1101444, 28)
                                                 0                     1  \
LOS-APP-ID                    100007_O01_341666128  100009_O01_341666137   
ioi_total_inquiries                              3                     1   
ioi_inq_last_3m                                  1                     0   
ioi_inq_last_6m                                  1                     0   
ioi_inq_last_12m                                 2                     1   
ioi_days_since_last_inq                         68                   211   
ioi_days_since_first_inq                       582                   211   
ioi_unique_lenders                               1                     1   
ioi_mfi_inq_count                                0                     1   
ioi_other_lender_inq_count                       3                     1   
ioi_max_inq_amt                            15000.0                   0.0   
ioi_total_inq_amt                          45000.0        

In [51]:
ioi_features['as_of_month'] = '2025-09-30'

In [48]:
# Load Inquiry (demographics) - set1 + set2
inquiry1 = load_bureau_csv(os.path.join(SET1_DIR, 'Inquiry.csv'))
inquiry2 = load_bureau_csv(os.path.join(SET2_DIR, 'Inquiry.csv'))

print(f'Inquiry set1: {inquiry1.shape}, set2: {inquiry2.shape}')

inquiry = pd.concat([inquiry1, inquiry2], ignore_index=True)
del inquiry1, inquiry2

# Drop trailing empty column
inquiry = inquiry.loc[:, ~(inquiry.columns.str.strip() == '')]

print(f'Inquiry combined: {inquiry.shape}')
print(f'Duplicates on LOS-APP-ID: {inquiry["LOS-APP-ID"].duplicated().sum()}')

Inquiry set1: (999998, 32), set2: (550373, 32)
Inquiry combined: (1550371, 32)
Duplicates on LOS-APP-ID: 0


In [52]:
# Base: Summary (1:1 per LOS-APP-ID)
# LEFT JOIN Account (1:many — this expands to trade-line level)
# Then LEFT JOIN IOI agg, Inquiry demog, Demog pivot (all 1:1 per LOS-APP-ID)

print('Starting joins...')
# print(f'  Summary base:      {summary.shape[0]:>12,} rows (1:1 per LOS-APP-ID)')
print(f'  Account:           {account_all.shape[0]:>12,} rows (1:many per LOS-APP-ID)')
print(f'  IOI agg:           {ioi_features.shape[0]:>12,} rows (1:1 per LOS-APP-ID)')
print(f'  Inquiry demog:     {inquiry.shape[0]:>12,} rows (1:1 per LOS-APP-ID)')
# print(f'  Demog pivot:       {demog_pivot.shape[0]:>12,} rows (1:1 per LOS-APP-ID)')

Starting joins...
  Account:             17,073,601 rows (1:many per LOS-APP-ID)
  IOI agg:              1,101,444 rows (1:1 per LOS-APP-ID)
  Inquiry demog:        1,550,371 rows (1:1 per LOS-APP-ID)


In [54]:
%%time
# Step 1: Summary LEFT JOIN Account (expands to account-level grain)
# master = summary.merge(account, on='LOS-APP-ID', how='left')
# print(f'After Summary + Account join: {master.shape[0]:,} rows')
# del account

# Step 2: LEFT JOIN IOI aggregated
master = account_all.merge(ioi_features,on='LOS-APP-ID', how='left')
print(f'After + IOI join: {master.shape[0]:,} rows')
# del ioi_agg
# # Step 4: LEFT JOIN Demog pivot
master = master.merge(inquiry, on='LOS-APP-ID', how='left')
print(f'After + Demog join: {master.shape[0]:,} rows')
# # del demog_pivot

After + IOI join: 17,073,601 rows
After + Demog join: 17,073,601 rows
CPU times: user 3.9 s, sys: 1.11 s, total: 5.01 s
Wall time: 5.04 s


In [56]:
# Extract loan_number, applicant_type, person_id from LOS-APP-ID
master = extract_los_parts(master)

# Add human-readable applicant role label
applicant_role_map = {'O01': 'Primary Applicant', 'O02': 'Co-Applicant', 'O03': 'Guarantor'}
master['applicant_role'] = master['applicant_type'].map(applicant_role_map)

print(f'\nFinal master shape: {master.shape}')
print(f'Unique LOS-APP-IDs: {master["LOS-APP-ID"].nunique():,}')
print(f'Unique loan_numbers: {master["loan_number"].nunique():,}')
print(f'\nApplicant type distribution:')
print(master[['applicant_type', 'applicant_role']].value_counts())
# print(f'\nRows with no Account data (NaN trade lines): {master["acct_acct_type"].isna().sum():,}')


Final master shape: (17073601, 99)
Unique LOS-APP-IDs: 1,550,287
Unique loan_numbers: 501,022

Applicant type distribution:
applicant_type  applicant_role   
O02             Co-Applicant         8980990
O01             Primary Applicant    6506607
O03             Guarantor            1586004
Name: count, dtype: int64


In [58]:
# QC checks
print('=== QC CHECKS ===')

# 1. No LOS-APP-IDs lost from Summary
# summary_ids = set(summary['LOS-APP-ID'].unique())
# master_ids = set(master['LOS-APP-ID'].unique())
# lost_ids = summary_ids - master_ids
# print(f'1. LOS-APP-IDs in Summary not in master: {len(lost_ids)} (should be 0)')

# # 2. All LOS-APP-IDs from Summary are present
# print(f'2. Summary LOS-APP-IDs: {len(summary_ids):,}, Master LOS-APP-IDs: {len(master_ids):,}')

# 3. Null coverage
null_pct = master.isnull().mean().sort_values(ascending=False)
print(f'\n3. Top 10 columns by null %:')
print((null_pct.head(10) * 100).round(1))

# 4. Column count
print(f'\n4. Total columns: {master.shape[1]}')

# del summary  # free memory

=== QC CHECKS ===

3. Top 10 columns by null %:
FATHER                100.0
ADDRESS-2             100.0
BRANCH-ID             100.0
CUSTOMER ID/MBR ID    100.0
OTHER                 100.0
MOTHER                100.0
PHONE 3               100.0
AKA                   100.0
PHONE 2               100.0
PASSPORT              100.0
dtype: float64

4. Total columns: 99


In [61]:
master.head(1).shape

(1, 99)

In [63]:
null_cols = master.columns[master.isna().all()].tolist()
print("Dropping columns:", null_cols)
master = master.drop(columns=null_cols)

Dropping columns: ['BRANCH-ID', 'CUSTOMER ID/MBR ID', 'AKA', 'PASSPORT', 'DL', 'OTHER-ID', 'FATHER', 'MOTHER', 'OTHER', 'PHONE 2', 'PHONE 3', 'EMAIL 1', 'EMAIL 2', 'ADDRESS-2', 'STATE-2', 'ADDRESS-3', 'STATE-3', 'RETRO-PR-DT', 'Unnamed: 31']


In [64]:
master.shape

(17073601, 80)

In [65]:
master.columns

Index(['CREDT-RPT-ID_x', 'LOS-APP-ID', 'CANDIDATE - ID', 'BRANCH', 'KENDRA',
       'SELF-INDICATOR', 'MATCH-TYPE', 'ACC-NUM', 'CREDIT-GRANTOR',
       'ACCT-TYPE', 'CONTRIBUTOR-TYPE', 'DATE-REPORTED', 'OWNERSHIP-IND',
       'ACCOUNT-STATUS', 'DISBURSED-DT', 'CLOSE-DT', 'LAST-PAYMENT-DATE',
       'CREDIT-LIMIT/SANC AMT', ' DISBURSED-AMT/HIGH CREDIT',
       ' INSTALLMENT-AMT', ' CURRENT-BAL', 'INSTALLMENT-FREQUENCY',
       'WRITE-OFF-DATE', ' OVERDUE-AMT', ' WRITE-OFF-AMT', 'ASSET_CLASS',
       'REPORTED DATE - HIST ', 'DPD - HIST', 'HIGH CRD - HIST ',
       'CUR BAL - HIST ', 'AMT OVERDUE - HIST ', 'AMT PAID - HIST ', 'INCOME ',
       ' INCOME INDICATOR ', 'TENURE ', ' OCCUPATION', 'ioi_total_inquiries',
       'ioi_inq_last_3m', 'ioi_inq_last_6m', 'ioi_inq_last_12m',
       'ioi_days_since_last_inq', 'ioi_days_since_first_inq',
       'ioi_unique_lenders', 'ioi_mfi_inq_count', 'ioi_other_lender_inq_count',
       'ioi_max_inq_amt', 'ioi_total_inq_amt', 'ioi_avg_inq_amt',
      

## Section 7: Save & Report

In [66]:
OUTPUT_DIR

'/Users/shiva/Downloads/Five_star_customer_data/bureau_data'

In [68]:
# Check which object columns are causing issues
print("Object dtype columns in master:")
print(master.select_dtypes(include="object").columns.tolist())

# Broad fix: cast all bool-like object cols to int
bool_like_cols = [col for col in master.columns 
                  if master[col].dtype == object 
                  and master[col].dropna().apply(lambda x: isinstance(x, bool)).any()]

print("Bool-infected columns:", bool_like_cols)
master[bool_like_cols] = master[bool_like_cols].astype(float).astype("Int64")  # nullable int

Object dtype columns in master:
['CREDT-RPT-ID_x', 'LOS-APP-ID', 'CANDIDATE - ID', 'BRANCH', 'KENDRA', 'SELF-INDICATOR', 'MATCH-TYPE', 'ACC-NUM', 'CREDIT-GRANTOR', 'ACCT-TYPE', 'CONTRIBUTOR-TYPE', 'DATE-REPORTED', 'OWNERSHIP-IND', 'ACCOUNT-STATUS', 'DISBURSED-DT', 'CLOSE-DT', 'LAST-PAYMENT-DATE', 'CREDIT-LIMIT/SANC AMT', ' DISBURSED-AMT/HIGH CREDIT', ' INSTALLMENT-AMT', ' CURRENT-BAL', 'INSTALLMENT-FREQUENCY', 'WRITE-OFF-DATE', ' OVERDUE-AMT', ' WRITE-OFF-AMT', 'ASSET_CLASS', 'REPORTED DATE - HIST ', 'DPD - HIST', 'HIGH CRD - HIST ', 'CUR BAL - HIST ', 'AMT OVERDUE - HIST ', 'AMT PAID - HIST ', 'INCOME ', ' INCOME INDICATOR ', 'TENURE ', ' OCCUPATION', 'ioi_fs_has_jlg_inq', 'ioi_fs_has_mfl_inq', 'as_of_month', 'CREDT-RPT-ID_y', 'APPLICANT-NAME', 'DOB', 'PAN', 'RC', 'UID', 'VOTERS', 'SPOUSE', 'GENDER', 'ADDRESS-1', 'STATE-1', 'loan_number', 'applicant_type', 'person_id', 'applicant_role']
Bool-infected columns: ['ioi_fs_has_jlg_inq', 'ioi_fs_has_mfl_inq']


In [69]:
%%time
# Save parquet (recommended for large data)
parquet_path = os.path.join(OUTPUT_DIR, 'bureau_master_account_level.parquet')
master.to_parquet(parquet_path, index=False)
parquet_size = os.path.getsize(parquet_path) / (1024**3)
print(f'Saved: {parquet_path}')
print(f'  Size: {parquet_size:.2f} GB')

# Save CSV
csv_path = os.path.join(OUTPUT_DIR, 'bureau_master_account_level.csv')
master.to_csv(csv_path, index=False)
csv_size = os.path.getsize(csv_path) / (1024**3)
print(f'Saved: {csv_path}')
print(f'  Size: {csv_size:.2f} GB')

Saved: /Users/shiva/Downloads/Five_star_customer_data/bureau_data/bureau_master_account_level.parquet
  Size: 2.68 GB
Saved: /Users/shiva/Downloads/Five_star_customer_data/bureau_data/bureau_master_account_level.csv
  Size: 15.20 GB
CPU times: user 4min 3s, sys: 15.4 s, total: 4min 18s
Wall time: 4min 26s


In [70]:
# # Generate report
# report_path = os.path.join(REPORT_DIR, 'bureau_master_creation_report.xlsx')

# with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
    
#     # Sheet 1: Overview
#     overview = pd.DataFrame({
#         'Metric': [
#             'Total rows', 'Unique LOS-APP-IDs', 'Unique loan_numbers',
#             'Unique person_ids', 'Total columns',
#             'Rows with Account data', 'Rows without Account data',
#             'Parquet file size (GB)', 'CSV file size (GB)',
#             'Creation timestamp'
#         ],
#         'Value': [
#             f'{master.shape[0]:,}',
#             f'{master["LOS-APP-ID"].nunique():,}',
#             f'{master["loan_number"].nunique():,}',
#             f'{master["person_id"].nunique():,}',
#             str(master.shape[1]),
#             f'{master["acct_acct_type"].notna().sum():,}',
#             f'{master["acct_acct_type"].isna().sum():,}',
#             f'{parquet_size:.2f}',
#             f'{csv_size:.2f}',
#             datetime.now().strftime('%Y-%m-%d %H:%M:%S')
#         ]
#     })
#     overview.to_excel(writer, sheet_name='Overview', index=False)
    
#     # Sheet 2: Null Analysis
#     null_analysis = pd.DataFrame({
#         'Column': master.columns,
#         'Dtype': master.dtypes.astype(str).values,
#         'Null_Count': master.isnull().sum().values,
#         'Null_Pct': (master.isnull().mean() * 100).round(2).values,
#         'Non_Null_Count': master.notna().sum().values
#     }).sort_values('Null_Pct', ascending=False)
#     null_analysis.to_excel(writer, sheet_name='Null Analysis', index=False)
    
#     # Sheet 3: Descriptive Stats (numeric columns only)
#     desc_stats = master.describe(include='number').T.reset_index()
#     desc_stats.rename(columns={'index': 'Column'}, inplace=True)
#     desc_stats.to_excel(writer, sheet_name='Descriptive Stats', index=False)
    
#     # Sheet 4: Applicant Type Distribution
#     app_type_dist = master.groupby('applicant_type').agg(
#         total_rows=('LOS-APP-ID', 'count'),
#         unique_applicants=('LOS-APP-ID', 'nunique'),
#         unique_loans=('loan_number', 'nunique'),
#         avg_trade_lines=('acct_acct_type', lambda x: x.notna().sum() / max(x.index.nunique(), 1)),
#         avg_cns_score=('summary_perform_cns_score', 'mean'),
#     ).reset_index()
#     app_type_dist.to_excel(writer, sheet_name='Applicant Type Dist', index=False)

# print(f'Report saved: {report_path}')

In [71]:
# # Final summary dashboard
# print('=' * 70)
# print('BUREAU MASTER DATA - CREATION SUMMARY')
# print('=' * 70)
# print(f'Grain: LOS-APP-ID x Bureau Account (trade line)')
# print(f'')
# print(f'Total rows:           {master.shape[0]:>15,}')
# print(f'Total columns:        {master.shape[1]:>15,}')
# print(f'Unique LOS-APP-IDs:   {master["LOS-APP-ID"].nunique():>15,}')
# print(f'Unique loan_numbers:  {master["loan_number"].nunique():>15,}')
# print(f'Unique person_ids:    {master["person_id"].nunique():>15,}')
# print(f'')
# print(f'Applicant types:')
# for (atype, role), count in master[['applicant_type', 'applicant_role']].value_counts().items():
#     print(f'  {atype} ({role}): {count:>12,} rows')
# print(f'')
# print(f'With bureau trade lines:    {master["acct_acct_type"].notna().sum():>12,}')
# print(f'Without (no accounts):      {master["acct_acct_type"].isna().sum():>12,}')
# print(f'')
# print(f'Output files:')
# print(f'  {parquet_path} ({parquet_size:.2f} GB)')
# print(f'  {csv_path} ({csv_size:.2f} GB)')
# print(f'  {report_path}')
# print('=' * 70)